# Eksperimen E3: Fine-Tuning Sentence-BERT untuk Retail Products
**Course**: Machine Learning  
**Dataset**: `06_retail_products_classification` (2.100 produk ritel, 21 kategori)  
**Panduan**: Sesuai Seksi 7 (Fine-tuning SBERT) & Seksi 11 (Eksperimen E3) pada `read.md`

### Outline Eksperimen:
1. Memuat model Sentence-BERT yang telah di-fine-tune (`models/sbert_finetuned`).
2. Menghitung embedding produk dan menjalankan pencarian semantik teradaptasi domain ritel.
3. Membandingkan performa 3 generasi model: **E1 (TF-IDF)** vs **E2 (SBERT Pretrained)** vs **E3 (SBERT Fine-Tuned)**.

In [1]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('../data/products.csv')
queries = pd.read_csv('../data/test_queries.csv')
df['doc'] = df['category'] + ': ' + df['item_name'] + '. ' + df['description'].fillna('')

print(f"Total Produk: {len(df)} item")
print(f"Total Query Uji: {len(queries)} query")

C:\Users\farha\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total Produk: 2100 item
Total Query Uji: 21 query


## 1. Memuat Model Hasil Fine-Tuning (`models/sbert_finetuned`)

In [2]:
model_path = '../models/sbert_finetuned'
finetuned_model = SentenceTransformer(model_path)

# Ekstraksi Embedding Produk dengan Model Fine-Tuned
corpus_embeddings = finetuned_model.encode(
    df['doc'].tolist(),
    batch_size=64,
    normalize_embeddings=True
)
print(f"Embedding Shape: {corpus_embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14206.29it/s]


Embedding Shape: (2100, 384)


## 2. Fungsi Pencarian Semantik Produk Ritel Teradaptasi

In [3]:
def search_finetuned(query, top_k=5):
    q_emb = finetuned_model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, corpus_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    
    results = []
    for rank, idx in enumerate(top_idx, 1):
        row = df.iloc[idx]
        results.append({
            'Rank': rank,
            'Title': row['item_name'],
            'Category': row['category'],
            'Similarity': round(float(sims[idx]), 4),
            'Image': row['image_path']
        })
    return pd.DataFrame(results)

# Uji coba kueri bebas
search_finetuned('power tools rotary hammer for workshop', top_k=3)

,Rank,Title,Category,Similarity,Image
0,1,Milwaukee 5303-20 1-1/8-Inch SDS-Plus Rotary H...,Tools & Home Improvement,0.4592,train/train/B00002247N.jpg
1,2,Makita HR3851 10 Amp 1-1/2-Inch Spline Rotary ...,Tools & Home Improvement,0.4374,train/train/B00002269U.jpg
2,3,Dremel 1680 1.4 Amp 16-Inch 1/6-Horsepower Ben...,Tools & Home Improvement,0.4319,train/train/B0000302YQ.jpg


## 3. Evaluasi Komparasi Tiga Tahap: E1 vs E2 vs E3

In [4]:
# Hitung metrik evaluasi model fine-tuned
k_list = [1, 3, 5]
recalls = {k: [] for k in k_list}
mrrs = []

for _, row in queries.iterrows():
    q = row['query']
    target = row['positive_item']
    
    q_emb = finetuned_model.encode([q], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, corpus_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:5]
    retrieved = [df.iloc[i]['item_name'] for i in top_idx]
    
    rank = 0
    for r_idx, item in enumerate(retrieved, 1):
        if item == target:
            rank = r_idx
            break
    mrrs.append(1.0 / rank if rank > 0 else 0.0)
    
    for k in k_list:
        recalls[k].append(1.0 if target in retrieved[:k] else 0.0)

comparison_table = pd.DataFrame({
    'Metrik Evaluasi': ['Recall@1 (Top-1 Match)', 'Recall@3 (Top-3 Match)', 'Recall@5 (Top-5 Match)', 'MRR (Ranking Quality)'],
    'E1: TF-IDF Baseline': ['71.43%', '90.48%', '90.48%', '0.8095'],
    'E2: SBERT Pretrained': ['76.19%', '85.71%', '95.24%', '0.8310'],
    'E3: SBERT Fine-Tuned': [f"{np.mean(recalls[1])*100:.2f}%", f"{np.mean(recalls[3])*100:.2f}%", f"{np.mean(recalls[5])*100:.2f}%", f"{np.mean(mrrs):.4f}"],
    'Total Gain vs E1': ['+14.28%', '+9.52%', '+9.52%', '+0.1191']
})

print("=== TABEL PERBANDINGAN LENGKAP E1 vs E2 vs E3 ===")
comparison_table

=== TABEL PERBANDINGAN LENGKAP E1 vs E2 vs E3 ===


,Metrik Evaluasi,E1: TF-IDF Baseline,E2: SBERT Pretrained,E3: SBERT Fine-Tuned,Total Gain vs E1
0,Recall@1 (Top-1 Match),71.43%,76.19%,85.71%,+14.28%
1,Recall@3 (Top-3 Match),90.48%,85.71%,100.00%,+9.52%
2,Recall@5 (Top-5 Match),90.48%,95.24%,100.00%,+9.52%
3,MRR (Ranking Quality),0.8095,0.8310,0.9286,+0.1191
